# Alternative optima in dFBA: diagnosis and parsimonious resolution

Figures for the reviewer comment on alternative optima.

The argument has three parts, and the panels follow them:

1. **Degeneracy is system-dependent, and we measured it rather than assuming it.** Of three
   published use cases, one is heavily degenerate and two have provably unique optima.
2. **Where it exists, it is spatially structured** — concentrated where the tissue is
   nutrient-poor, not spread uniformly.
3. **pFBA resolves it without changing the phenotype.** Growth is preserved; what changes is
   the flux distribution, and specifically the byproducts and amino acids rather than the
   primary carbon and oxygen uptakes.

Panel D is the direct consequence for the model: the exchange fluxes *are* the coupling to
BioFVM, so a non-unique flux distribution means a non-unique extracellular environment.

## Data provenance

Every panel reads from `experiments_fva/`, which holds the completed runs and their analyses.
See each experiment's own `README.md` for how it was produced and for its caveats — in
particular `bacterial_colony_600min/README.md` explains why its routine self-check misreports
and why the result nonetheless stands.

| panel | source |
| --- | --- |
| A | the three `fva_summary.csv` files |
| B, S1 | `cancer_core_metabolism_4320min/fva_cells.csv` |
| C, D | `cancer_core_metabolism_4320min{,_pfba}/output` |

The two cancer runs use byte-identical configurations apart from the `<secondary_objective>`
block, and the same `omp_num_threads`. That last point matters: PhysiCell seeds a `thread_local`
generator from a per-thread table sized by `omp_num_threads`, so runs at different thread counts
are not comparable.

In [ ]:
import os, glob
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..", ".."))
EXP  = os.path.join(ROOT, "experiments_fva")
OUT  = os.path.join(EXP, "figures_reviewer_response")
os.makedirs(OUT, exist_ok=True)
assert os.path.isdir(EXP), f"experiments_fva not found at {EXP}"

# --- palette -------------------------------------------------------------
# Okabe-Ito, reordered so the weakest colour-vision-deficient adjacent pair is
# not adjacent. Checked rather than assumed: lightness band, chroma floor, CVD
# separation and normal-vision separation all pass; the orange sits at 2.19:1
# against white, which is why every series that uses it is directly labelled
# rather than relying on a legend swatch.
#
# This figure deliberately commits to a light/print look, so no dark variant.
BLUE, VERM, GREEN, ORANGE, PURPLE = "#0072B2", "#D55E00", "#009E73", "#E69F00", "#8B5FA8"
INK, MUTED, GRID = "#1a1a1a", "#6b6b6b", "#dcdcdc"
SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#f7fbff", BLUE, "#02304a"])

mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300,
    "font.size": 8.5, "axes.titlesize": 9.5, "axes.labelsize": 8.5,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "legend.frameon": False, "figure.facecolor": "white", "axes.facecolor": "white",
    "lines.linewidth": 1.6,
})

def nice(rid):
    """R_EX_lac_L_e -> lactate-style short label."""
    s = rid.replace("R_EX_", "").replace("_e", "")
    s = s.replace("__D", "").replace("__L", "").replace("_L", "").replace("_D", "")
    return {"glc": "glucose", "o2": "O$_2$", "co2": "CO$_2$", "nh4": "NH$_4^+$",
            "lac": "lactate", "ac": "acetate", "pi": "phosphate", "chol": "choline",
            "gln": "glutamine", "glu": "glutamate", "gly": "glycine", "orn": "ornithine",
            "asn": "asparagine", "asp": "aspartate", "thr": "threonine", "ser": "serine",
            "arg": "arginine", "val": "valine", "lys": "lysine", "ile": "isoleucine",
            "phe": "phenylalanine", "trp": "tryptophan", "tyr": "tyrosine",
            "leu": "leucine"}.get(s, s)
print("root:", ROOT)

## Loading

`load_snapshot` reconstructs the labelled cell matrix from a MultiCellDS pair. The label list has
to be expanded by each entry's `size` attribute, because vector quantities such as `position`
occupy three consecutive rows under a single label.

In [ ]:
def load_snapshot(stem):
    """stem is a path without extension, e.g. .../output/final"""
    cn = ET.parse(stem + ".xml").getroot().find(".//cell_population/custom/simplified_data")
    labels = []
    for L in cn.find("labels").findall("label"):
        n, nm = int(L.get("size")), L.text.strip()
        labels += [nm] if n == 1 else [f"{nm}[{i}]" for i in range(n)]
    mat = sio.loadmat(os.path.join(os.path.dirname(stem), cn.find("filename").text))["cells"]
    return pd.DataFrame({l: mat[i, :] for i, l in enumerate(labels)})

def exchanges_of(df):
    return sorted(c for c in df.columns
                  if c.startswith("R_EX_") and not c.endswith(("_lb", "_ub", "_range",
                                                               "_fva_min", "_fva_max")))

CANCER_FBA  = os.path.join(EXP, "cancer_core_metabolism_4320min")
CANCER_PFBA = os.path.join(EXP, "cancer_core_metabolism_4320min_pfba")

fba  = load_snapshot(os.path.join(CANCER_FBA,  "output", "final"))
pfba = load_snapshot(os.path.join(CANCER_PFBA, "output", "final"))
EX = exchanges_of(fba)
print(f"cancer slice at 4320 min: FBA {len(fba)} cells, pFBA {len(pfba)} cells, {len(EX)} exchanges")
print("note: the cell sets are NOT identical -- over 4320 min the trajectories diverge through")
print("      BioFVM feedback and division stochasticity, so C and D compare depth-binned")
print("      distributions rather than matched cells.")

## Panel A — degeneracy is system-dependent

The job here is to compare magnitudes across three systems whose exchange sets are *different*
(23 for the cancer slice, 5 and 4 for the two *E. coli* cases), so a shared categorical axis
would not work. One row per system, one dot per exchange.

A boxplot would be the wrong form: two of the three systems are at exactly zero, and a box drawn
at zero reads as missing data rather than as a result. Dots on a visible axis make "nothing here"
look deliberate.

In [ ]:
USE_CASES = [
    ("Tumour slice\n(MCF7 core)",        "cancer_core_metabolism_4320min", BLUE),
    ("Acetate switch\n(E. coli core)",   "ecoli_acetic_switch_600min",     VERM),
    ("Bacterial colony\n(E. coli core)", "bacterial_colony_600min",        GREEN),
]
summaries = {name: pd.read_csv(os.path.join(EXP, d, "fva_summary.csv"))
             for name, d, _ in USE_CASES}
for name, s in summaries.items():
    print(f"{name.splitlines()[0]:20s} {len(s):2d} exchanges, "
          f"max relative range {np.nanmax(s.median_rel.values):.3g}")

In [ ]:
def panel_A(ax):
    for row, (name, _, colour) in enumerate(USE_CASES):
        s = summaries[name]
        v = np.nan_to_num(s["median_rel"].to_numpy(float), nan=0.0)
        y = np.full(len(v), row) + np.linspace(-0.13, 0.13, len(v))
        ax.scatter(v, y, s=26, color=colour, alpha=0.85, zorder=3,
                   edgecolor="white", linewidth=0.5)
        # name the three widest, which are the ones the argument rests on
        order = np.argsort(v)[::-1]
        # Stagger the three labels: at these values they sit close together on x
        # and identically on y, so a fixed offset prints them over each other.
        for n, k in enumerate(order[:3]):
            if v[k] <= 1e-9:
                break
            ax.annotate(nice(s["reaction"].iloc[k]), (v[k], y[k]),
                        textcoords="offset points",
                        xytext=(6, 12 - 11 * n), fontsize=7.5, color=INK,
                        arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.5,
                                        shrinkA=0, shrinkB=2))
        if v.max() <= 1e-9:
            ax.annotate("every exchange uniquely determined", (0, row),
                        textcoords="offset points", xytext=(26, 0), va="center",
                        fontsize=7.5, style="italic", color=MUTED)
    ax.set_yticks(range(len(USE_CASES)))
    ax.set_yticklabels([n for n, _, _ in USE_CASES])
    ax.set_ylim(len(USE_CASES) - 0.5, -0.5)
    ax.set_xlabel("relative degeneracy   (FVA interval width / |flux|)")
    ax.set_title("A   Alternative optima are system-dependent", loc="left", weight="bold")
    ax.axvline(0, color=MUTED, lw=0.8, zorder=2)
    ax.grid(axis="y", visible=False)
    ax.margins(x=0.14)

fig, ax = plt.subplots(figsize=(6.0, 2.5)); panel_A(ax); plt.tight_layout(); plt.show()

## Panel B — and where it exists, it is spatially structured

Cancer slice only, since it is the only system with any degeneracy. The five widest exchanges
are drawn as a median line with an interquartile ribbon against distance from the nutrient
boundary; the remaining eighteen are summarised as a single grey band so the panel does not
carry twenty-three colours.

Each series is directly labelled at its right-hand end rather than relying on a legend, which is
also what makes the lighter hues legible.

In [ ]:
cells_csv = os.path.join(CANCER_FBA, "fva_cells.csv")
head = pd.read_csv(cells_csv, nrows=0).columns.tolist()
need = ["time", "position[0]", "growth_rate"]
need += [c for e in EX for c in (e, f"{e}_range") if c in head]
# 348 MB: take every 4th row, which is ample for medians over depth bins
cdf = pd.read_csv(cells_csv, usecols=[c for c in need if c in head],
                  skiprows=lambda i: i > 0 and i % 4 != 0)
print(f"{len(cdf):,} cell-timepoints sampled, t = {cdf.time.min():g} to {cdf.time.max():g} min")

# Absolute interval width, in flux units. Deliberately NOT the relative measure
# panel A uses: dividing per-row by a small |flux| inflates the ratio without
# bound, and having "relative degeneracy" mean two different things in two
# panels of one figure is worse than carrying two units.
rel = pd.DataFrame({e: np.abs(cdf[f"{e}_range"].to_numpy(float))
                    for e in EX if f"{e}_range" in cdf})
rank = rel.median(skipna=True).sort_values(ascending=False)
TOP5 = [e for e in rank.index[:5]]
print("widest five:", [nice(e) for e in TOP5])

In [ ]:
DEPTH_EDGES = np.arange(0, 361, 20)
DEPTH_MID = 0.5 * (DEPTH_EDGES[:-1] + DEPTH_EDGES[1:])
depth_bin = np.digitize(cdf["position[0]"].to_numpy(float), DEPTH_EDGES) - 1

def band(values):
    med = np.full(len(DEPTH_MID), np.nan)
    lo  = np.full(len(DEPTH_MID), np.nan)
    hi  = np.full(len(DEPTH_MID), np.nan)
    for b in range(len(DEPTH_MID)):
        v = values[(depth_bin == b) & np.isfinite(values)]
        if v.size >= 20:
            med[b], lo[b], hi[b] = np.median(v), *np.percentile(v, [25, 75])
    return med, lo, hi

def panel_B(ax):
    others = [e for e in rel.columns if e not in TOP5]
    if others:
        om, ol, oh = band(rel[others].to_numpy(float).ravel()
                          if False else np.nanmedian(rel[others].to_numpy(float), axis=1))
        ax.fill_between(DEPTH_MID, ol, oh, color=GRID, alpha=0.9, lw=0, zorder=1)
        ax.plot(DEPTH_MID, om, color=MUTED, lw=1.0, zorder=2)
        k = np.where(np.isfinite(om))[0]
        if k.size:
            ax.annotate(f"other {len(others)} exchanges (median)",
                        (DEPTH_MID[k[0]], om[k[0]]), textcoords="offset points",
                        xytext=(6, -11), ha="left", fontsize=7, color=MUTED)
    ends = []
    for e, colour in zip(TOP5, [BLUE, VERM, GREEN, ORANGE, PURPLE]):
        m, lo, hi = band(rel[e].to_numpy(float))
        ax.fill_between(DEPTH_MID, lo, hi, color=colour, alpha=0.16, lw=0, zorder=3)
        ax.plot(DEPTH_MID, m, color=colour, zorder=4)
        k = np.where(np.isfinite(m))[0]
        if k.size:
            ends.append((m[k[-1]], DEPTH_MID[k[-1]], nice(e), colour))
    # The five series converge at the right edge, so labels placed at their own
    # end values land on top of each other. Spread them vertically, keeping
    # their order, and leave the colour to tie each back to its line.
    ends.sort(reverse=True)
    span = ax.get_ylim()[1] - ax.get_ylim()[0]
    step = span * 0.075
    prev = None
    for val, xend, label, colour in ends:
        y = val if prev is None else min(val, prev - step)
        prev = y
        ax.annotate(label, (xend, val), xytext=(xend + 7, y),
                    textcoords="data", va="center", fontsize=7.5,
                    color=colour, weight="bold",
                    arrowprops=dict(arrowstyle="-", color=colour, lw=0.6,
                                    shrinkA=0, shrinkB=1, alpha=0.6))
    ax.set_xlabel("distance from the nutrient boundary  ($\mu$m)")
    ax.set_ylabel("FVA interval width  (mmol gDW$^{-1}$ h$^{-1}$)")
    ax.set_title("B   The ambiguity concentrates away from the nutrient source",
                 loc="left", weight="bold")
    ax.set_xlim(0, 360)
    ymax = ax.get_ylim()[1]
    ax.axvspan(195, 360, color=GRID, alpha=0.35, lw=0, zorder=0)
    ax.annotate("no growing cells beyond this depth\n(cf. panel C)",
                (278, ymax * 0.62), ha="center", va="center",
                fontsize=7, color=MUTED, style="italic")

fig, ax = plt.subplots(figsize=(6.0, 2.8)); panel_B(ax); plt.tight_layout(); plt.show()

## Panel C — growth is preserved

The control for the whole argument. The band nearest the nutrient boundary is held by a Dirichlet
condition, so microenvironment feedback cannot reach it: if pFBA preserved growth anywhere it
must do so there. Deeper bands are *expected* to differ, because a parsimonious cell consumes
less and leaves more for the cells behind it — that is the coupling working, not an error.

In [ ]:
def depth_profile(df, col, edges=DEPTH_EDGES):
    x = df["position[0]"].to_numpy(float)
    v = df[col].to_numpy(float)
    b = np.digitize(x, edges) - 1
    mid, mean, sem = [], [], []
    for k in range(len(edges) - 1):
        s = v[b == k]
        if s.size:
            mid.append(0.5 * (edges[k] + edges[k + 1]))
            mean.append(s.mean())
            sem.append(s.std(ddof=1) / np.sqrt(s.size) if s.size > 1 else 0.0)
    return np.array(mid), np.array(mean), np.array(sem)

def panel_C(ax):
    for df, name, colour, dash in [(fba, "standard FBA", BLUE, (0, (4, 2))),
                                   (pfba, "pFBA", VERM, None)]:
        x, m, s = depth_profile(df, "growth_rate")
        ax.plot(x, m, color=colour, linestyle=dash if dash else "-", label=name, zorder=3)
        ax.fill_between(x, m - s, m + s, color=colour, alpha=0.18, lw=0, zorder=2)
    ax.set_xlabel("distance from the nutrient boundary  ($\mu$m)")
    ax.set_ylabel("growth rate  (min$^{-1}$)")
    ax.set_title("C   Growth is preserved where feedback cannot reach",
                 loc="left", weight="bold")
    # Two series only: the legend carries identity and no direct labels are
    # needed. The first version annotated both at the same point and printed
    # one name on top of the other.
    ax.legend(loc="upper right")
    ax.set_xlim(0, 360)

fig, ax = plt.subplots(figsize=(6.0, 2.6)); panel_C(ax); plt.tight_layout(); plt.show()

# the quantitative claim behind the panel
for lo in range(0, 360, 60):
    mf = fba[(fba["position[0]"] >= lo) & (fba["position[0]"] < lo + 60)]["growth_rate"]
    mp = pfba[(pfba["position[0]"] >= lo) & (pfba["position[0]"] < lo + 60)]["growth_rate"]
    if len(mf) and len(mp):
        d = f"{100*(mp.mean()/mf.mean()-1):+6.2f}%" if mf.mean() > 1e-12 else "    n/a"
        print(f"  {lo:3d}-{lo+60:3d} um   FBA {mf.mean():.7g}   pFBA {mp.mean():.7g}   {d}")

## Panel D — what pFBA actually changes

Direction matters here and zero is meaningful, so this is a diverging form: $\log_2$ of the
ratio of mean absolute flux, sorted, with a neutral midpoint.

An exchange is plotted only if **both** runs carry flux above an absolute floor. A ratio needs a
meaningful denominator as well as a meaningful numerator: glutamate moves from
$1.7\times10^{-4}$ to $1.4\times10^{-3}$ mmol/gDW/h, which is a ratio of 8.4 and would be the
largest bar in the panel, but both values are effectively zero against a glucose uptake of
$2.5\times10^{-2}$. Excluded exchanges are listed below the figure rather than silently dropped,
and the floor is stated on the panel.

In [ ]:
D_FLOOR = 1e-3   # mmol/gDW/h; below this in both runs the ratio is not meaningful

# A ratio is only meaningful when BOTH runs carry real flux, so require both
# above the floor -- not just one of them. Requiring only max(u, v) keeps cases
# like glutamate (1.7e-4 -> 1.4e-3) whose ratio of 8.4 is an artefact of a
# near-zero denominator, and that bar would dominate the panel.
rows, excluded = [], []
for e in EX:
    u, v = np.abs(fba[e].to_numpy(float)).mean(), np.abs(pfba[e].to_numpy(float)).mean()
    if min(u, v) < D_FLOOR:
        excluded.append((e, u, v))
        continue
    rows.append((e, u, v, np.log2(v / u)))
dd = pd.DataFrame(rows, columns=["reaction", "fba", "pfba", "log2"]).sort_values("log2")
dropped = len(excluded)

def panel_D(ax):
    y = np.arange(len(dd))
    colours = [VERM if v < 0 else BLUE for v in dd["log2"]]
    ax.barh(y, dd["log2"], color=colours, height=0.68, zorder=3)
    ax.set_yticks(y); ax.set_yticklabels([nice(r) for r in dd["reaction"]], fontsize=7.5)
    ax.axvline(0, color=INK, lw=1.0, zorder=4)
    ax.set_xlabel("$\\log_2$ ( pFBA / standard FBA )   of mean |flux|")
    ax.set_title("D   Byproducts and amino acids move; the primary uptakes do not",
                 loc="left", weight="bold")
    ax.grid(axis="y", visible=False)
    for lbl, xpos, ha in [("less flux", -0.08, "right"), ("more flux", 0.08, "left")]:
        ax.annotate(lbl, (xpos, len(dd) - 0.2), xycoords=("data", "data"),
                    fontsize=7, color=MUTED, ha=ha, style="italic")
    ax.annotate(f"{dropped} exchanges below {D_FLOOR:g} mmol/gDW/h in both runs, not shown",
                (0.5, -0.22), xycoords="axes fraction", ha="center",
                fontsize=7, color=MUTED)

fig, ax = plt.subplots(figsize=(6.0, 4.2)); panel_D(ax); plt.tight_layout(); plt.show()
print(dd.assign(ratio=lambda d: d.pfba / d.fba)[["reaction", "fba", "pfba", "ratio"]]
        .to_string(index=False))
print(f"\nexcluded ({D_FLOOR:g} mmol/gDW/h floor on BOTH runs) -- ratios here are not meaningful:")
for e, u, v in sorted(excluded, key=lambda t: -max(t[1], t[2])):
    print(f"  {e:14s} FBA {u:.3e}  pFBA {v:.3e}  (ratio {v/max(u,1e-18):.2f})")

## The assembled figure

Panels B and C share the depth axis so the reader can carry a position across them: where the
ambiguity is largest is also where growth is lowest.

In [ ]:
fig = plt.figure(figsize=(11.0, 7.4))
gs = fig.add_gridspec(2, 2, width_ratios=[1.25, 1.0], height_ratios=[1.0, 1.15],
                      hspace=0.42, wspace=0.28)
axA = fig.add_subplot(gs[0, 0]); panel_A(axA)
axB = fig.add_subplot(gs[1, 0]); panel_B(axB)
axC = fig.add_subplot(gs[0, 1]); panel_C(axC)
axD = fig.add_subplot(gs[1, 1]); panel_D(axD)
for fmt in ("png", "pdf"):
    p = os.path.join(OUT, f"fig_alternative_optima.{fmt}")
    fig.savefig(p, bbox_inches="tight", facecolor="white")
    print("wrote", p)
plt.show()

## Supplementary S1 — does degeneracy grow as the slice becomes hypoxic?

The time dimension, which the main figure leaves out. Exchange by simulated time, coloured by
the median FVA interval width — the same absolute measure panel B uses, not panel A's
relative one — on a single sequential hue — one hue, light to dark, never a rainbow,
so that "more" reads unambiguously as "darker".

Rows are ordered by overall degeneracy, so the block structure is the result: a handful of
byproducts carry essentially all of it, and they do so from early on rather than only once the
core goes hypoxic.

In [ ]:
TIME_EDGES = np.linspace(cdf.time.min(), cdf.time.max(), 37)
tb = np.digitize(cdf.time.to_numpy(float), TIME_EDGES) - 1
order = [e for e in rank.index if e in rel.columns]

M = np.full((len(order), len(TIME_EDGES) - 1), np.nan)
for i, e in enumerate(order):
    v = rel[e].to_numpy(float)
    for j in range(len(TIME_EDGES) - 1):
        s = v[(tb == j) & np.isfinite(v)]
        if s.size >= 10:
            M[i, j] = np.median(s)

fig, ax = plt.subplots(figsize=(7.4, 5.0))
im = ax.imshow(M, aspect="auto", cmap=SEQ, origin="upper",
               extent=[TIME_EDGES[0], TIME_EDGES[-1], len(order) - 0.5, -0.5],
               vmin=0, vmax=np.nanpercentile(M, 99))
ax.set_yticks(range(len(order)))
ax.set_yticklabels([nice(e) for e in order], fontsize=7.5)
ax.set_xlabel("simulated time  (min)")
ax.set_title("S1   Degeneracy by exchange and time, tumour slice", loc="left", weight="bold")
ax.grid(visible=False)
cb = fig.colorbar(im, ax=ax, pad=0.02, fraction=0.045)
cb.set_label("FVA interval width  (mmol gDW$^{-1}$ h$^{-1}$, median)")
cb.outline.set_visible(False)
for fmt in ("png", "pdf"):
    p = os.path.join(OUT, f"figS1_degeneracy_vs_time.{fmt}")
    fig.savefig(p, bbox_inches="tight", facecolor="white"); print("wrote", p)
plt.show()

## Numbers for the response letter

In [ ]:
print("Degeneracy by use case (relative interval width, median over cell-timepoints)")
for name, d, _ in USE_CASES:
    s = summaries[name]
    v = np.nan_to_num(s["median_rel"].to_numpy(float), nan=0.0)
    w = s["max_range"].to_numpy(float)
    tag = "NO degeneracy" if np.nanmax(np.abs(w)) < 1e-9 else f"max {v.max():.3g}"
    print(f"  {name.replace(chr(10),' '):36s} {len(s):2d} exchanges   {tag}")

print("\nTumour slice, pFBA vs standard FBA at 4320 min")
print(f"  mean growth           {fba.growth_rate.mean():.6g} -> {pfba.growth_rate.mean():.6g}"
      f"  ({100*(pfba.growth_rate.mean()/fba.growth_rate.mean()-1):+.2f}%)")
tf = sum(np.abs(fba[e]).mean()  for e in EX)
tp = sum(np.abs(pfba[e]).mean() for e in EX)
print(f"  total |exchange flux| {tf:.6g} -> {tp:.6g}  ({100*(tp/tf-1):+.2f}%)")
print(f"  cells                 {len(fba)} -> {len(pfba)}")